In [12]:
import objaverse.xl as oxl
import pandas as pd
import re
import json

all_annotations = oxl.get_annotations(
    download_dir="~/.objaverse" # default download directory
)


# Apply preprocessing once for all rows
annotations = all_annotations[all_annotations["metadata"]!="{}"]
annotations["description"] = annotations["metadata"].map(json.loads).map(lambda d: d.get("filename", d.get("title")).replace("_", " "))


/tmp/ipykernel_19480/1739183445.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations["description"] = annotations["metadata"].map(json.loads).map(lambda d: d.get("filename", d.get("title")).replace("_", " "))


In [16]:
all_annotations[all_annotations["source"]=="sketchfab"].keys()

Index(['fileIdentifier', 'source', 'license', 'fileType', 'sha256',
       'metadata'],
      dtype='object')

In [22]:
# print complete values
all_annotations[all_annotations["source"]=="sketchfab"].sample(10)["fileIdentifier"].values

array(['https://sketchfab.com/3d-models/5c5849935fb549db878a7b00dbcab216',
       'https://sketchfab.com/3d-models/24cc1c25dccb4fdb906d5ae988dee40d',
       'https://sketchfab.com/3d-models/e87f69be7b8b4412aa0564e0a7575fe8',
       'https://sketchfab.com/3d-models/e1deb581292348798ace0efd09d835f5',
       'https://sketchfab.com/3d-models/4c1204fc4d7e4e03901064af1897bb7a',
       'https://sketchfab.com/3d-models/5f19e69e2b464f4789fc5838e163d5eb',
       'https://sketchfab.com/3d-models/d6ff370fc21e4927ba2a9b626a22d126',
       'https://sketchfab.com/3d-models/7b10570974d04dc6ba5f9e9222dbce73',
       'https://sketchfab.com/3d-models/528e8741520041da92a8be9f7ca93724',
       'https://sketchfab.com/3d-models/395deaf13a1f4d6bbe02a137f6529dfb'],
      dtype=object)

In [ ]:
# source
# github         5236361
# sketchfab       796031
# smithsonian       2407
# thingiverse    3732212
# dtype: int64

In [10]:

# set cookie for thingverse
import os
import shutil
import subprocess
from tqdm import tqdm

os.environ["THINGIVERSE_COOKIE"] = "PHPSESSID=1107b3c7ca0a2a02f0f74aac180b583b; _cfuvid=7aNzA9oe1fO4DFQcjzLYcEgkxhIlxCwr9HcHUitooSs-1768517254442-0.0.1.1-604800000; previous_url=%2F; _sharedID=2aad3c88-7dd4-48bb-bb31-69489e181e93; _sharedID_cst=zix7LPQsHA%3D%3D; CookieConsent={stamp:%27-1%27%2Cnecessary:true%2Cpreferences:true%2Cstatistics:true%2Cmarketing:true%2Cmethod:%27implied%27%2Cver:3%2Cutc:1768518124396%2Ciab2:%27%27%2Cregion:%27US-26%27}; _sharedid=851923a1-fa1e-4344-a526-9ff35dbcab93; _sharedid_cst=VyxHLMwsHQ%3D%3D; cf_clearance=rXvMGcqVvCVZ_nTF5a9ZGaaVOfOmhIkVIr0B1lF7aRw-1768518126-1.2.1.1-I.Mc06RPA1DVsaI98eeC5yBmRONXENWdaimJ25kooEaVrkHMVPk0m0tmnK7cEfa12E6EeQ_qmDdcxQ4.pvYN6fvjBoISv5GXrsMJWVZYuBiUN2XdmI3eL2UovKEtsX.fG3gxHw66_Z_ph53RXoLjX16kDyk8uDDH7SEUcEUfIclBLfRzQfWe3ZdrecKXjTwFUkf8xpVJtAYG6fUaAzDVZfNJ4YD9.hao4DJn7dg9Xj0; hb_insticator_uid=173809e2-00ce-4d7a-813a-7b43f230a7dc; _lr_sampling_rate=0; _lr_retry_request=true; _lr_env_src_ats=false; __gads=ID=1b6747a78124fa89:T=1768518129:RT=1768518129:S=ALNI_MZNeU31N93K4Y2jrw4va3FtADdVMg; __gpi=UID=000013293e453757:T=1768518129:RT=1768518129:S=ALNI_MYorZ_j1SiizWr29YuHyn7NOJwxxQ; __eoi=ID=682673b74c613af4:T=1768518129:RT=1768518129:S=AA-AfjaAn0UyiMIr2tKHRg---FMi; _awl=2.1768518304.5-c9b9d345ce9110e35a0d53a8aeb7a180-6763652d75732d63656e7472616c31-1; cto_bundle=UreeDl9NemVOTVJMZVpkTDNmZXNpNzdmcFExeG9yYnE1VXd1c0tBdklNdSUyRkVVczBKSXolMkJYaXA1Sk55M2FraDclMkIwcFllckhZWEdYayUyRklLY3ZIQ3NSVU45eEdkSGtIaHhQaVhDaTYzUVBZSWFHZ3hDTDRZVlkyJTJCYVl3cHBXWFNuWGR2bSUyRndtbkJEeGR5cmlzdnBiV1NpTjUySlElM0QlM0Q; cto_bidid=6b7ru19rT1BORnBNOWt1ZGJuaVpmZm42NEEzYklscldrcXZnSHNwYzd6JTJGdzg5cGFvMjBKYVVhMFdXQmJtJTJCZDFsRkw3azF2bU10VmQlMkJpZVJLck5ZZFpiZlR0N3VaNEJvS2dLOVF3ZXRWWGF1QWVpT0tGZWdYVFlBV3ZCdml5JTJCZXlNUmdU"
os.environ["THINGIVERSE_USER_AGENT"] = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36"

# Example keyword
keyword = "cat"

# check if keyword is in description
mask = annotations["description"].apply(lambda x: " "+keyword.lower()+" " in x or x.endswith(" "+keyword.lower()) or x.startswith(keyword.lower()+" "))
annotations_matched = annotations[mask]
# print(annotations_matched)

# # show fileIdentifier, description, type, source
print(annotations_matched[["fileIdentifier", "description", "fileType", "source"]])
# aggregate by filetype
print("fileType", annotations_matched.groupby("fileType").size())
print("source", annotations_matched.groupby("source").size())

annotations_matched = annotations_matched.iloc[:30]
# shutil.rmtree(os.path.expanduser("~/.objaverse"), ignore_errors=True)


                                            fileIdentifier  \
5242461  https://thingiverse.com/thing:3589966/files?fi...   
5242462  https://thingiverse.com/thing:3589966/files?fi...   
5242463  https://thingiverse.com/thing:3589966/files?fi...   
5242464  https://thingiverse.com/thing:3589966/files?fi...   
5242465  https://thingiverse.com/thing:3589966/files?fi...   
...                                                    ...   
8947552  https://thingiverse.com/thing:1213827/files?fi...   
8952675  https://thingiverse.com/thing:6150199/files?fi...   
8960430  https://thingiverse.com/thing:6129928/files?fi...   
8961408  https://thingiverse.com/thing:5793698/files?fi...   
8967085  https://thingiverse.com/thing:6133618/files?fi...   

                               description fileType       source  
5242461  cat lying down-bs3-remix-bank.stl      stl  thingiverse  
5242462            cat lying down-plug.stl      stl  thingiverse  
5242463             cat lying down-key.stl      stl  t

In [ ]:

# Download objects using the new curl_cffi backed downloader
downloaded_objects = oxl.download_objects(
    objects=annotations_matched,
)

# Process downloaded objects
# Save in @robot_env/tools/objaverse/data
output_base_dir = os.path.abspath("data")
os.makedirs(output_base_dir, exist_ok=True)

# Path to blender script (assumed to be in same directory as this notebook)
blender_script = os.path.abspath("render_blender.py")

print(f"Downloaded {len(downloaded_objects)} objects. Processing...")

# Iterate and process
# Convert dict items to list to guarantee order if needed, though dict preserves insertion order in py3.7+
for i, (file_identifier, local_path) in enumerate(tqdm(downloaded_objects.items())):
    # Create folder name {keyword}_{number}
    folder_name = f"{keyword}_{i}"
    object_dir = os.path.join(output_base_dir, folder_name)
    os.makedirs(object_dir, exist_ok=True)
    
    # Define destination path
    ext = os.path.splitext(local_path)[1]
    # Handle files without extension if any, though usually they have one
    if not ext:
        ext = ".stl" # Default for thingiverse
    
    dest_path = os.path.join(object_dir, f"object{ext}")
    
    # Copy file to destination
    if os.path.exists(local_path):
        shutil.copy2(local_path, dest_path)
    else:
        print(f"Warning: Source file not found: {local_path}")
        continue

    if not os.path.exists(dest_path):
        print(f"Error: copy file failed")
    else:
        print(f"Processed {dest_path}")
    
    # Render images using Blender
    # We call blender in background mode
    # Assumes /snap/bin/blender exists as per check
    blender_executable = "/snap/bin/blender"
    if not os.path.exists(blender_executable):
        # try just "blender"
        blender_executable = "blender"
    
    blender_cmd = [
        blender_executable,
        "--background",
        "--python", blender_script,
        "--",
        "--input", dest_path,
        "--output_dir", object_dir
    ]

    
    try:
        # Run blender, suppress output unless error
        subprocess.run(blender_cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Failed to render {folder_name}: {e}")
        # print stderr for debugging
        if e.stderr:
            print(e.stderr.decode())
    except FileNotFoundError:
        print("Blender not found. Please install blender or ensure it is in PATH.")
        break

print(f"Done processing. Data saved in {output_base_dir}")
